In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification, ViTImageProcessor, ViTConfig
from peft import get_peft_model, LoraConfig
from tqdm import tqdm
import time
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score
from torch.cuda.amp import autocast, GradScaler

# ===========================
# Step 0: Dataset Verification
# ===========================
def verify_dataset(path):
    dataset = ImageFolder(path)
    class_names = dataset.classes
    print(f"Found {len(class_names)} classes in {path}:")
    print(class_names)
    return len(class_names)

# Verify and load class counts
train_classes = verify_dataset("imd_temp/train")
val_classes = verify_dataset("imd_temp/val")
test_classes = verify_dataset("imd_temp/test")
# assert train_classes == val_classes == test_classes, "Class mismatch between datasets!"
num_classes = train_classes

# ===========================
# Step 1: Load Pretrained ViT with Correct Config
# ===========================
model_name = "google/vit-base-patch16-224-in21k"

# Load and modify config
config = ViTConfig.from_pretrained(model_name)
config.patch_embeddings_bias = False
config.num_labels = num_classes  # Set labels in config

# Load model with updated config
model = ViTForImageClassification.from_pretrained(
    model_name,
    config=config,
    ignore_mismatched_sizes=True  # Keep this for classification head resizing
)

# ===========================
# Step 2: Apply LoRA Adaptation
# ===========================
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["key", "value"],
    modules_to_save=["classifier"]
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ===========================
# Step 3: Data Preparation
# ===========================
processor = ViTImageProcessor.from_pretrained(model_name)
transform = transforms.Compose([
    transforms.Resize((processor.size["height"], processor.size["width"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

# Create datasets with validation
train_dataset = ImageFolder("imd_temp/train", transform=transform)
val_dataset = ImageFolder("imd_temp/val", transform=transform)
test_dataset = ImageFolder("imd_temp/test", transform=transform)

# DataLoader with pinned memory
def create_loader(dataset, shuffle):
    return DataLoader(
        dataset,
        batch_size=4,
        shuffle=shuffle,
        num_workers=2,
        persistent_workers=True,
        pin_memory=True
    )

train_loader = create_loader(train_dataset, shuffle=True)
val_loader = create_loader(val_dataset, shuffle=False)
test_loader = create_loader(test_dataset, shuffle=False)

# ===========================
# Step 4: Training Setup
# ===========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# # Label validation function
# def check_labels(labels):
#     assert torch.all(labels >= 0) and torch.all(labels < num_classes), \
#         f"Invalid labels detected! Max: {labels.max()}, Min: {labels.min()}"

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = GradScaler()
accumulation_steps = 2

# Metrics with proper device placement
metrics = {
    "accuracy": MulticlassAccuracy(num_classes=num_classes).to(device),
    "precision": MulticlassPrecision(num_classes=num_classes, average="macro").to(device),
    "recall": MulticlassRecall(num_classes=num_classes, average="macro").to(device),
    "f1_score": MulticlassF1Score(num_classes=num_classes, average="macro").to(device),
}

# ===========================
# Step 5: Enhanced Training Loop
# ===========================
best_val_loss = float("inf")
patience = 3
counter = 0
num_epochs = 20

for epoch in range(num_epochs):
    # Training Phase
    model.train()
    train_loss = 0
    epoch_start = time.time()
    
    optimizer.zero_grad()
    for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")):
        # Data validation
        # check_labels(labels)
        
        # Data transfer with async
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        # Mixed precision forward
        with autocast():
            outputs = model(pixel_values=images, labels=labels)
            loss = outputs.loss
            
        # Gradient accumulation
        scaler.scale(loss / accumulation_steps).backward()
        
        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        train_loss += loss.item()

    # Validation Phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(pixel_values=images, labels=labels)
            val_loss += outputs.loss.item()
            
            preds = outputs.logits.argmax(dim=1)
            for metric in metrics.values():
                metric.update(preds, labels)

    # Calculate epoch statistics
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    epoch_time = time.time() - epoch_start
    
    print(f"\nEpoch {epoch+1}/{num_epochs} - {epoch_time:.1f}s")
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"Val Accuracy: {metrics['accuracy'].compute().item():.4f}")
    print(f"Val F1-Score: {metrics['f1_score'].compute().item():.4f}")
    
    # Reset metrics
    for metric in metrics.values():
        metric.reset()

    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "vit_lora_r8.pth")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered!")
            break

# ===========================
# Step 6: Final Evaluation
# ===========================
model.load_state_dict(torch.load("vit_lora_r8.pth"))
model.eval()

test_results = {}
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        outputs = model(pixel_values=images)
        preds = outputs.logits.argmax(dim=1)
        
        for name, metric in metrics.items():
            metric.update(preds, labels)
    
    # Collect final metrics
    for name, metric in metrics.items():
        test_results[name] = metric.compute().item()

print("\nFinal Test Results:")
for name, value in test_results.items():
    print(f"{name.capitalize()}: {value:.4f}")

# Save final model
torch.save(model.state_dict(), "vit_lora_r8.pth")
print("Model saved successfully!")

d:\anaconda\envs\pyTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 10 classes in imd_temp/train:
['GB_80', 'GB_MF_80', 'GB_RS_80', 'MF_80', 'MF_GB_80', 'MF_RS_80', 'Original_80', 'RS_80', 'RS_GB_80', 'RS_MF_80']
Found 10 classes in imd_temp/val:
['GB_80', 'GB_MF_80', 'GB_RS_80', 'MF_80', 'MF_GB_80', 'MF_RS_80', 'Original_80', 'RS_80', 'RS_GB_80', 'RS_MF_80']
Found 10 classes in imd_temp/test:
['GB_80', 'GB_MF_80', 'GB_RS_80', 'MF_80', 'MF_GB_80', 'MF_RS_80', 'Original_80', 'RS_80', 'RS_GB_80', 'RS_MF_80']


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 302,602 || all params: 86,108,948 || trainable%: 0.3514


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14508\980920051.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1/20 [Train]:   0%|          | 0/75000 [00:00<?, ?it/s]C:\Users\ASUS\AppData\Local\Temp\ipykernel_14508\980920051.py:137: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1/20 [Val]: 100%|██████████| 15000/15000 [13:31<00:00, 18.48it/s]



Epoch 1/20 - 4897.6s
Train Loss: 0.4030 | Val Loss: 0.2944
Val Accuracy: 0.8744
Val F1-Score: 0.8744


... (10 outputs left out)

Epoch 7/20 [Val]: 100%|██████████| 15000/15000 [13:30<00:00, 18.50it/s]
C:\Users\ASUS\AppData\Local\Temp\ipykernel_14508\980920051.py:194: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental


Epoch 7/20 - 4901.1s
Train Loss: 0.2160 | Val Loss: 0.1952
Val Accuracy: 0.9195
Val F1-Score: 0.9195
Early stopping triggered!


Testing: 100%|██████████| 25000/25000 [22:35<00:00, 18.44it/s]



Final Test Results:
Accuracy: 0.9223
Precision: 0.9230
Recall: 0.9223
F1_score: 0.9221
Model saved successfully!


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification, ViTImageProcessor, ViTConfig
from peft import get_peft_model, LoraConfig
from tqdm import tqdm
import time
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score
from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===========================
# Step 0: Dataset Verification
# ===========================
def verify_dataset(path):
    dataset = ImageFolder(path)
    class_names = dataset.classes
    print(f"Found {len(class_names)} classes in {path}:")
    print(class_names)
    return len(class_names)

# Verify and load class counts
# train_classes = verify_dataset("imd_temp/train")
# val_classes = verify_dataset("imd_temp/val")
test_classes = verify_dataset("imd_temp/test")
# assert train_classes == val_classes == test_classes, "Class mismatch between datasets!"
num_classes = test_classes

# ===========================
# Step 1: Load Pretrained ViT with Correct Config
# ===========================
model_name = "google/vit-base-patch16-224-in21k"

# Load and modify config
config = ViTConfig.from_pretrained(model_name)
config.num_labels = num_classes  # Set labels in config

# Load model with updated config
model = ViTForImageClassification.from_pretrained(
    model_name,
    config=config,
    ignore_mismatched_sizes=True  # Keep this for classification head resizing
).to(device)

# ===========================
# Step 2: Apply LoRA Adaptation (Optional)
# ===========================
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    modules_to_save=["classifier"]
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ===========================
# Step 3: Data Preparation
# ===========================
processor = ViTImageProcessor.from_pretrained(model_name)
transform = transforms.Compose([
    transforms.Resize((processor.size["height"], processor.size["width"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

test_dataset = ImageFolder("imd_temp/test", transform=transform)

# DataLoader with pinned memory
def create_loader(dataset, shuffle):
    return DataLoader(
        dataset,
        batch_size=32,
        shuffle=shuffle,
        num_workers=2,
        persistent_workers=True,
        pin_memory=True
    )

test_loader = create_loader(test_dataset, shuffle=False)

# ===========================
# Step 4: Load Trained Model
# ===========================
model.load_state_dict(torch.load("vit_lora_final.pth", map_location=device))
model.eval()

# Define evaluation metrics
metrics = {
    "accuracy": MulticlassAccuracy(num_classes=num_classes).to(device),
    "precision": MulticlassPrecision(num_classes=num_classes, average="macro").to(device),
    "recall": MulticlassRecall(num_classes=num_classes, average="macro").to(device),
    "f1_score": MulticlassF1Score(num_classes=num_classes, average="macro").to(device),
}

# ===========================
# Step 5: Testing Loop
# ===========================
test_results = {}

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        preds = outputs.logits.argmax(dim=1)

        for name, metric in metrics.items():
            metric.update(preds, labels)

    # Collect final metrics
    for name, metric in metrics.items():
        test_results[name] = metric.compute().item()

print("\nFinal Test Results:")
for name, value in test_results.items():
    print(f"{name.capitalize()}: {value:.4f}")

# # ===========================
# # Step 6: Save Final Model
# # ===========================
# torch.save(model.state_dict(), "vit_lora_final.pth")
# print("Model saved successfully!")


Found 10 classes in imd_temp/test:
['GB_80', 'GB_MF_80', 'GB_RS_80', 'MF_80', 'MF_GB_80', 'MF_RS_80', 'Original_80', 'RS_80', 'RS_GB_80', 'RS_MF_80']


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 155,146 || all params: 85,961,492 || trainable%: 0.1805


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14508\1482416192.py:89: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("vit_lora_final.pth", map_l